# 面试问题：Temporal RAG 怎样处理 Valid Time、发布时间和冲突引用？

可以直接复述的回答是：第一，文档必须记录规则何时生效，而不仅是抓取时间。第二，回答“当时是什么规则”还要限制系统在那个时间已知的文档。第三，检索时按 topic、valid interval 和 published_at 三重过滤。第四，重叠且值不同的规则应输出冲突而非任选一条。第五，修订文档通过 supersedes 链取代旧版本。第六，答案引用必须带有效区间和知识时间。下面用差旅与远程办公政策演示。

## 真实案例：员工查询不同日期的政策标准

知识库含 8 条版本化政策，覆盖餐补、远程办公和酒店上限。五个问题分别询问 2025/2026 不同日期，并带“系统截至何时已知”的 known_at。文档和金额为教学构造，不代表真实公司制度。

In [1]:
documents = [  # 定义八条带双时间和修订关系的政策片段
    {"id": "M1", "topic": "meal", "value": "100元/天", "valid_from": "2025-01-01", "valid_to": "2025-12-31", "published_at": "2024-12-20", "revision": 1, "supersedes": None},  # 2025 餐补
    {"id": "M2", "topic": "meal", "value": "120元/天", "valid_from": "2026-01-01", "valid_to": None, "published_at": "2025-12-15", "revision": 2, "supersedes": None},  # 2026 餐补
    {"id": "R1", "topic": "remote", "value": "每周2天", "valid_from": "2025-01-01", "valid_to": "2025-06-30", "published_at": "2024-12-28", "revision": 1, "supersedes": None},  # 上半年远程规则
    {"id": "R2", "topic": "remote", "value": "每周3天", "valid_from": "2025-07-01", "valid_to": None, "published_at": "2025-06-20", "revision": 2, "supersedes": None},  # 下半年远程规则
    {"id": "R3", "topic": "remote", "value": "每周4天", "valid_from": "2025-09-01", "valid_to": None, "published_at": "2025-08-25", "revision": 1, "supersedes": None},  # 未声明修订关系的冲突规则
    {"id": "H1", "topic": "hotel", "value": "600元/晚", "valid_from": "2025-01-01", "valid_to": None, "published_at": "2024-12-10", "revision": 1, "supersedes": None},  # 初始酒店上限
    {"id": "H2", "topic": "hotel", "value": "650元/晚", "valid_from": "2025-07-01", "valid_to": None, "published_at": "2026-02-01", "revision": 2, "supersedes": "H1"},  # 事后发布的追溯修订
    {"id": "T1", "topic": "taxi", "value": "夜间实报实销", "valid_from": "2025-01-01", "valid_to": None, "published_at": "2024-12-18", "revision": 1, "supersedes": None},  # 无冲突的出租车规则
]  # 结束版本化政策库
questions = [  # 定义五条带 valid time 和 knowledge time 的问题
    {"id": "TR-01", "topic": "meal", "as_of": "2025-03-10", "known_at": "2026-07-01", "expected": "100元/天"},  # 回看 2025 餐补
    {"id": "TR-02", "topic": "meal", "as_of": "2026-07-10", "known_at": "2026-07-10", "expected": "120元/天"},  # 当前 2026 餐补
    {"id": "TR-03", "topic": "remote", "as_of": "2025-05-01", "known_at": "2025-05-01", "expected": "每周2天"},  # 上半年远程规则
    {"id": "TR-04", "topic": "hotel", "as_of": "2025-08-01", "known_at": "2025-08-15", "expected": "600元/晚"},  # 当时尚未知追溯修订
    {"id": "TR-05", "topic": "remote", "as_of": "2025-10-01", "known_at": "2025-10-01", "expected": "conflict"},  # 两条重叠远程规则冲突
]  # 结束五个时间问题
print("政策输入：id | topic | value | valid | published | supersedes")  # 展示 Temporal RAG 的双时间字段
for document in documents:  # 逐条输出八个版本
    print(f"{document['id']} | {document['topic']:6} | {document['value']:8} | {document['valid_from']}..{document['valid_to'] or '∞'} | {document['published_at']} | {document['supersedes'] or '-'}")  # 呈现有效期与发布时间差异
print("查询集：", [(item["id"], item["as_of"], item["known_at"], item["expected"]) for item in questions])  # 展示五个历史或当前问题


政策输入：id | topic | value | valid | published | supersedes
M1 | meal   | 100元/天   | 2025-01-01..2025-12-31 | 2024-12-20 | -
M2 | meal   | 120元/天   | 2026-01-01..∞ | 2025-12-15 | -
R1 | remote | 每周2天     | 2025-01-01..2025-06-30 | 2024-12-28 | -
R2 | remote | 每周3天     | 2025-07-01..∞ | 2025-06-20 | -
R3 | remote | 每周4天     | 2025-09-01..∞ | 2025-08-25 | -
H1 | hotel  | 600元/晚   | 2025-01-01..∞ | 2024-12-10 | -
H2 | hotel  | 650元/晚   | 2025-07-01..∞ | 2026-02-01 | H1
T1 | taxi   | 夜间实报实销   | 2025-01-01..∞ | 2024-12-18 | -
查询集： [('TR-01', '2025-03-10', '2026-07-01', '100元/天'), ('TR-02', '2026-07-10', '2026-07-10', '120元/天'), ('TR-03', '2025-05-01', '2025-05-01', '每周2天'), ('TR-04', '2025-08-01', '2025-08-15', '600元/晚'), ('TR-05', '2025-10-01', '2025-10-01', 'conflict')]


## Baseline / 基线：每个主题只取最新发布文档

最新文档不一定在问题日期生效，也可能在当时尚未发布。Baseline 对五题一律取 published_at 最大项。

In [2]:
def latest_baseline(topic):  # 返回主题下发布时间最新的文档
    candidates = [document for document in documents if document["topic"] == topic]  # 获取同主题所有版本
    return max(candidates, key=lambda document: document["published_at"])  # 完全忽略问题时间和冲突
baseline_rows = []  # 收集五题最新文档答案
print("最新文档 Baseline：id | doc | value | correct")  # 输出逐问题时间错误
for question in questions:  # 对五个 as_of 问题运行最新策略
    document = latest_baseline(question["topic"])  # 获取主题最新发布版本
    correct = document["value"] == question["expected"]  # 与人工历史答案比较
    baseline_rows.append((question["id"], document["id"], document["value"], correct))  # 保存基线结果
    print(f"{question['id']} | {document['id']} | {document['value']} | {correct}")  # 展示过去问题被未来版本覆盖


最新文档 Baseline：id | doc | value | correct
TR-01 | M2 | 120元/天 | False
TR-02 | M2 | 120元/天 | True
TR-03 | R3 | 每周4天 | False
TR-04 | H2 | 650元/晚 | False
TR-05 | R3 | 每周4天 | False


## 核心实现：双时间过滤、Supersedes 和冲突检测

候选必须在 as_of 有效且在 known_at 已发布。若新文档显式 supersedes 旧文档，移除旧版本；剩余多个不同值则返回 conflict 并列出全部引用。

In [3]:
def active_on(document, as_of):  # 判断规则在业务时间是否有效
    starts = document["valid_from"] <= as_of  # 检查查询日期不早于生效日
    not_ended = document["valid_to"] is None or as_of <= document["valid_to"]  # 检查查询日期不晚于终止日
    return starts and not_ended  # 返回 valid-time 区间命中
def temporal_retrieve(topic, as_of, known_at):  # 执行双时间检索和冲突解析
    candidates = [document for document in documents if document["topic"] == topic and active_on(document, as_of) and document["published_at"] <= known_at]  # 同时过滤主题、有效期和知识时间
    superseded_ids = {document["supersedes"] for document in candidates if document["supersedes"]}  # 收集候选中被明确取代的旧文档
    survivors = [document for document in candidates if document["id"] not in superseded_ids]  # 移除已被当前可知修订取代的版本
    values = {document["value"] for document in survivors}  # 汇总仍有效候选的不同规则值
    if len(values) == 1:  # 唯一规则值可安全回答
        return {"status": "answer", "value": next(iter(values)), "documents": survivors, "candidates": candidates}  # 返回答案和完整引用
    if len(values) > 1:  # 重叠规则给出不同值时不能任选
        return {"status": "conflict", "value": "conflict", "documents": survivors, "candidates": candidates}  # 返回冲突和并列证据
    return {"status": "not_found", "value": "not_found", "documents": [], "candidates": candidates}  # 无有效版本时明确拒答
focus = temporal_retrieve("hotel", "2025-08-01", "2025-08-15")  # 查询 2025 年 8 月当时可知的酒店上限
print("TR-04 候选：", [(document["id"], document["value"], document["published_at"]) for document in focus["candidates"]])  # 展示 H2 尚未发布因此不可见
print("TR-04 结果：", focus["value"], "citations=", [document["id"] for document in focus["documents"]])  # 展示历史答案和引用
conflict_focus = temporal_retrieve("remote", "2025-10-01", "2025-10-01")  # 查询重叠远程规则
print("TR-05 冲突：", [(document["id"], document["value"]) for document in conflict_focus["documents"]])  # 展示 R2 与 R3 的相反有效值


TR-04 候选： [('H1', '600元/晚', '2024-12-10')]
TR-04 结果： 600元/晚 citations= ['H1']
TR-05 冲突： [('R2', '每周3天'), ('R3', '每周4天')]


## 失败案例与修正：未来发布的追溯修订不能污染“当时已知”答案

H2 的 valid_from 是 2025-07，但到 2026-02 才发布。忽略 known_at 会在 2025-08 的历史回放中使用未来知识；双时间过滤保留当时可见的 H1。

In [4]:
def valid_time_only(topic, as_of):  # 模拟只检查有效期、不检查发布时间的错误检索
    candidates = [document for document in documents if document["topic"] == topic and active_on(document, as_of)]  # 允许未来发布文档进入过去回答
    return max(candidates, key=lambda document: document["revision"])  # 选择最高修订版本
future_leak_document = valid_time_only("hotel", "2025-08-01")  # 用错误方法回放 2025 年 8 月酒店规则
safe_history = temporal_retrieve("hotel", "2025-08-01", "2025-08-15")  # 使用当时知识边界重新检索
later_history = temporal_retrieve("hotel", "2025-08-01", "2026-03-01")  # 在修订发布后重建同一业务日期
print("忽略 knowledge time：", future_leak_document["id"], future_leak_document["value"], future_leak_document["published_at"])  # 展示未来文档泄漏
print("截至 2025-08 已知：", safe_history["value"], [document["id"] for document in safe_history["documents"]])  # 展示历史当时答案
print("截至 2026-03 已知：", later_history["value"], [document["id"] for document in later_history["documents"]])  # 展示追溯修订后的知识状态


忽略 knowledge time： H2 650元/晚 2026-02-01
截至 2025-08 已知： 600元/晚 ['H1']
截至 2026-03 已知： 650元/晚 ['H2']


## 结果表：五个时间问题的答案与引用

In [5]:
temporal_rows = []  # 收集五题双时间结果
print("id | expected | result | status | citations | correct")  # 输出逐问题时间检索表
for question in questions:  # 对五个问题使用同一双时间入口
    result = temporal_retrieve(question["topic"], question["as_of"], question["known_at"])  # 检索有效且当时已知的规则
    citations = [document["id"] for document in result["documents"]]  # 提取所有支持或冲突引用
    correct = result["value"] == question["expected"]  # 与人工历史答案或 conflict 比较
    temporal_rows.append({"id": question["id"], "value": result["value"], "status": result["status"], "citations": citations, "correct": correct})  # 保存可审计结果
    print(f"{question['id']} | {question['expected']} | {result['value']} | {result['status']} | {citations} | {correct}")  # 展示答案和时间证据
baseline_accuracy = sum(row[3] for row in baseline_rows) / len(baseline_rows)  # 计算最新文档策略准确率
temporal_accuracy = sum(row["correct"] for row in temporal_rows) / len(temporal_rows)  # 计算双时间策略准确率
print(f"准确率：latest={baseline_accuracy:.1%}，temporal={temporal_accuracy:.1%}；conflicts={sum(row['status'] == 'conflict' for row in temporal_rows)}")  # 汇总正确性和冲突发现


id | expected | result | status | citations | correct
TR-01 | 100元/天 | 100元/天 | answer | ['M1'] | True
TR-02 | 120元/天 | 120元/天 | answer | ['M2'] | True
TR-03 | 每周2天 | 每周2天 | answer | ['R1'] | True
TR-04 | 600元/晚 | 600元/晚 | answer | ['H1'] | True
TR-05 | conflict | conflict | conflict | ['R2', 'R3'] | True
准确率：latest=20.0%，temporal=100.0%；conflicts=1


## 结果解读

TR-01 不会被 2026 餐补覆盖，TR-04 也不会使用当时尚未发布的 H2。到了 2026-03，同一个 2025-08 业务日期因追溯修订而返回 H2，说明 valid time 与 knowledge time回答不同问题。R2/R3 没有 supersedes 关系且值冲突，因此系统列出两条引用并拒绝选择。

## 生产边界

生产 Temporal RAG 需要不可变文档版本、时区、闭开区间规范、撤销、追溯修订和事务时间数据库。生成答案必须带“适用日期”和“系统截至日期”，冲突进入制度管理员流程。向量检索只能召回候选，时间过滤必须在权威元数据层完成。本例日期使用可排序 ISO 字符串。

## 最小回归测试

In [6]:
assert len(documents) >= 5 and len(questions) >= 5  # 保证时间案例包含多个版本和问题
assert focus["value"] == "600元/晚" and [document["id"] for document in focus["documents"]] == ["H1"]  # 保证历史回答不泄漏未来修订
assert future_leak_document["id"] == "H2" and safe_history["value"] == "600元/晚"  # 保证失败案例可复现并被双时间修正
assert later_history["value"] == "650元/晚"  # 保证修订发布后 supersedes 生效
assert conflict_focus["status"] == "conflict" and len(conflict_focus["documents"]) == 2  # 保证重叠不同值不会被静默选择
assert temporal_accuracy > baseline_accuracy  # 保证双时间检索在同一五题上优于最新文档策略
